# 9. Jämföra två andelar (A/B-test)

I förra notebooken jämförde vi två **medelvärden**. Lika ofta handlar frågan om två **andelar**:
klickar fler på den nya knappen? Blir fler friska med den nya behandlingen? Har stödet för ett förslag ändrats?

Logiken är exakt densamma som förut. Bara formeln för medelfelet är ny.

**Efter den här notebooken kan du**
- formulera hypoteser för en jämförelse av två andelar
- genomföra ett z-test för två andelar, för hand och med scipy
- räkna ut ett konfidensintervall för skillnaden
- skilja på statistisk signifikans och praktisk betydelse

**Förkunskaper:** [8. Hypotesprövning och p-värden](../08_hypotesprovning/08_hypotesprovning.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

## 9.1 Ett A/B-test

En webbutik testar två versioner av sin startsida. Besökarna lottas till version A eller B, 1 000 till varje.
Vi räknar hur många som klickar på "Köp".

In [ ]:
clicks_a, n_a = 115, 1000
clicks_b, n_b = 150, 1000

p_a = clicks_a / n_a
p_b = clicks_b / n_b
print(f"Version A: {p_a:.1%}   Version B: {p_b:.1%}   Skillnad: {p_b - p_a:.1%}")

B fick 3,5 procentenheter fler klick. Men två grupper får aldrig exakt samma resultat, inte ens om sidorna är identiska.

- $H_0: p_A = p_B$ (versionerna är lika bra)
- $H_1: p_A \neq p_B$

## 9.2 Hur stor skillnad kan slumpen ge?

Precis som med myntet i förra notebooken börjar vi med att simulera en värld där $H_0$ är sann.
Anta att *båda* versionerna har samma verkliga klickfrekvens, och se hur stora skillnader som uppstår ändå.

🔮 **Gissa först:** om båda versionerna egentligen har 13 % klickfrekvens, hur ofta skiljer sig två grupper om 1 000
besökare med 3,5 procentenheter eller mer?

In [ ]:
n_per_group = 1000        # ← ändra: pröva 200 och 5000
true_rate = 0.1325        # samma klickfrekvens i båda grupperna (H0 sann)
observed_diff = 0.035

sim_a = rng.binomial(n_per_group, true_rate, 10_000) / n_per_group
sim_b = rng.binomial(n_per_group, true_rate, 10_000) / n_per_group
sim_diff = sim_b - sim_a

plt.hist(sim_diff, bins=50, color="lightgray", edgecolor="white")
plt.axvline(observed_diff, color="red", linestyle="--", label="observerad skillnad")
plt.axvline(-observed_diff, color="red", linestyle="--")
plt.xlabel("Skillnad i andel (B − A) när H0 är sann")
plt.ylabel("Antal simulerade experiment")
plt.title(f"n = {n_per_group} per grupp")
plt.legend()
plt.show()

print(f"Typisk slumpskillnad (standardavvikelse): {sim_diff.std():.4f}")
print(f"Andel simuleringar med minst {observed_diff:.1%} skillnad: {(np.abs(sim_diff) >= observed_diff).mean():.4f}")

Den sista raden är ett simulerat p-värde. En skillnad på 3,5 procentenheter uppstår sällan av ren slump när grupperna
har 1 000 besökare var.

🧪 **Labba:** Ändra `n_per_group` till 200 och kör om. Samma skillnad på 3,5 procentenheter – är den fortfarande ovanlig?
Pröva sedan 5 000.

## 9.3 z-testet

Simuleringen kan ersättas med en formel. Testvariabeln är som vanligt *skillnad / medelfel*:

$$z = \frac{\hat{p}_B - \hat{p}_A}{\sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_A} + \frac{1}{n_B}\right)}}$$

Här är $\hat{p}$ den **sammanvägda (poolade) andelen**: alla klick delat med alla besökare. Vi använder den eftersom
$H_0$ säger att det bara finns *en* gemensam klickfrekvens, och då är det den bästa skattningen av den.

In [ ]:
p_pool = (clicks_a + clicks_b) / (n_a + n_b)
std_error_diff = np.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
z = (p_b - p_a) / std_error_diff
p_value = 2 * stats.norm.sf(abs(z))

print(f"Sammanvägd andel:    {p_pool:.4f}")
print(f"Skillnadens medelfel: {std_error_diff:.4f}   (jämför med standardavvikelsen i simuleringen)")
print(f"z = {z:.2f}")
print(f"p-värde = {p_value:.4f}")

In [ ]:
alpha = 0.05
if p_value < alpha:
    print("Slutsats: förkasta H0. Andelen som klickar skiljer sig signifikant mellan versionerna.")
else:
    print("Slutsats: H0 kan inte förkastas. Vi saknar belägg för att versionerna skiljer sig.")

z = 2,31 betyder att den observerade skillnaden är 2,31 medelfel från noll. Från 68–95–99,7-regeln vet du att det är ovanligt
att hamna mer än 2 standardavvikelser från mitten, och p-värdet bekräftar det.

**Förutsättningar:** slumpmässig fördelning till grupperna, oberoende observationer och så stora stickprov att
normalfördelningen passar (tumregel: $n\hat{p}(1-\hat{p}) > 5$ i båda grupperna).

### Kontroll med scipy

scipy har ingen egen funktion för z-testet, men samma test kan göras som ett **chi-två-test** på tabellen över klick och
icke-klick. Det ger samma p-värde, och $z^2 = \chi^2$. Chi-två-testet får du lära dig i
[notebook 11](../11_korstabeller_chi2/11_korstabeller_chi2.ipynb).

In [ ]:
table = [[clicks_a, n_a - clicks_a],
         [clicks_b, n_b - clicks_b]]
res = stats.chi2_contingency(table, correction=False)
print(f"χ² = {res.statistic:.2f}  (z² = {z**2:.2f}),  p-värde = {res.pvalue:.4f}")

## 9.4 Hur stor är skillnaden? Konfidensintervall

p-värdet säger att skillnaden troligen inte är noll. Det säger inte hur stor den är. Det gör konfidensintervallet:

$$(\hat{p}_B - \hat{p}_A) \pm z^* \sqrt{\frac{\hat{p}_A(1-\hat{p}_A)}{n_A} + \frac{\hat{p}_B(1-\hat{p}_B)}{n_B}}$$

(Här används inte den sammanvägda andelen, eftersom intervallet inte utgår från att $H_0$ är sann.)

In [ ]:
diff = p_b - p_a
se_ci = np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
z_crit = stats.norm.ppf(0.975)

print(f"Skillnad: {diff:.1%}")
print(f"95 % KI:  [{diff - z_crit * se_ci:.1%}, {diff + z_crit * se_ci:.1%}]")

Intervallet innehåller inte 0, vilket stämmer med att testet blev signifikant. Men lägg märke till hur brett det är:
den sanna förbättringen kan vara allt från ungefär en halv till över sex procentenheter.

## 9.5 Samma andelar, olika mycket data

Funktionen nedan samlar hela analysen, så att du kan pröva egna siffror.

In [ ]:
def two_proportion_test(x_a, n_a, x_b, n_b):
    """z-test och 95 % KI för skillnaden mellan två andelar (B − A)."""
    p_a, p_b = x_a / n_a, x_b / n_b
    p_pool = (x_a + x_b) / (n_a + n_b)
    z = (p_b - p_a) / np.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
    p_value = 2 * stats.norm.sf(abs(z))
    margin = stats.norm.ppf(0.975) * np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
    print(f"A: {p_a:.1%}  B: {p_b:.1%}  skillnad: {p_b - p_a:+.1%}   z = {z:.2f}   p = {p_value:.4f}   "
          f"95 % KI: [{p_b - p_a - margin:+.1%}, {p_b - p_a + margin:+.1%}]")
    return z, p_value

two_proportion_test(115, 1000, 150, 1000);

✏️ **Din tur:** Anta exakt samma andelar (11,5 % och 15,0 %) men
1. bara 200 besökare per version (23 och 30 klick),
2. 10 000 besökare per version (1 150 och 1 500 klick).

Kör `two_proportion_test` för båda. Vad händer med p-värdet och med konfidensintervallet? Vad säger det om resultat
från små A/B-test?

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
two_proportion_test(23, 200, 30, 200)
two_proportion_test(1150, 10_000, 1500, 10_000);
# Med 200 per grupp är samma skillnad inte signifikant, och intervallet sträcker sig från klart negativt till klart positivt.
# Med 10 000 per grupp är intervallet smalt och p-värdet mycket litet.
# Ett litet test kan alltså missa en verklig effekt – "icke-signifikant" betyder inte "ingen skillnad".
```

</details>

## 9.6 Signifikant är inte samma sak som viktigt

Vänd på experimentet: vad händer med en *pytteliten* skillnad när man har väldigt mycket data?

🔮 **Gissa först:** version A får 10,0 % klick och version B 10,3 %, med en miljon besökare var. Signifikant?

In [ ]:
two_proportion_test(100_000, 1_000_000, 103_000, 1_000_000);

p-värdet är försvinnande litet: skillnaden är med största säkerhet *verklig*. Men 0,3 procentenheter kanske inte är värd
kostnaden att bygga om sajten. Det avgör inte statistiken, utan verksamheten.

> **Statistisk signifikans** svarar på: *är skillnaden mer än slump?*
> **Praktisk betydelse** svarar på: *är skillnaden stor nog att bry sig om?*
>
> Med lite data kan viktiga skillnader bli icke-signifikanta. Med mycket data blir oviktiga skillnader signifikanta.
> Redovisa därför alltid skillnadens **storlek** och ett **konfidensintervall**, inte bara p-värdet.

✏️ **Din tur:** Två vaccin jämförs. Av 4 000 personer som fick vaccin A blev 48 sjuka. Av 4 000 som fick vaccin B blev 30 sjuka.
1. Formulera hypoteserna.
2. Gör testet. Är skillnaden statistiskt säkerställd på 5 %-nivån?
3. Kontrollera tumregeln $n\hat{p}(1-\hat{p}) > 5$ för båda grupperna.

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
# H0: p_A = p_B (lika stor andel blir sjuka).  H1: p_A ≠ p_B.
two_proportion_test(48, 4000, 30, 4000)
for x, n in [(48, 4000), (30, 4000)]:
    print(f"n·p̂(1−p̂) = {n * (x / n) * (1 - x / n):.1f}")
# p ≈ 0,04 < 0,05: skillnaden är statistiskt säkerställd, men med liten marginal.
# Konfidensintervallets ena ände ligger nära 0. Tumregeln är uppfylld i båda grupperna.
```

</details>

## Sammanfattning

- Två andelar jämförs med ett **z-test**: z = skillnad / medelfel, där medelfelet räknas med den sammanvägda andelen.
- Samma procentuella skillnad kan vara signifikant eller inte, beroende på **stickprovsstorleken**.
- **Konfidensintervallet** för skillnaden visar hur stor effekten kan vara.
- **Signifikant ≠ viktigt.** Bedöm alltid effektens storlek i sitt sammanhang.

### Kontrollfrågor
1. Varför används den sammanvägda andelen i testet men inte i konfidensintervallet?
2. Ett A/B-test med 50 besökare per grupp visar 20 % mot 30 % och p = 0,25. Chefen säger: "Då spelar versionen ingen roll." Kommentera.

<details>
<summary>💡 <b>Svar</b></summary>

1. Testet räknar under antagandet att $H_0$ är sann, alltså att det finns en gemensam andel. Konfidensintervallet gör inget sådant antagande.
2. Testet är för litet för att säga något. Tio procentenheter är en stor skillnad *om* den är verklig, men med 50 per grupp
   är osäkerheten enorm. Slutsatsen är "vi vet inte", inte "ingen skillnad". Samla mer data.

</details>

**Nästa notebook:** [10. Typ I- och typ II-fel](../10_typ1_typ2_fel/10_typ1_typ2_fel.ipynb)